In [1]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import yaml
import os
import csv
import gc
from ultralytics import YOLO
import random
import torch
from glob import glob
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from ultralytics.utils.metrics import ConfusionMatrix

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
!cp -r /kaggle/input/rasoul-b-obb/dataset /kaggle/working/

In [4]:
!yolo settings runs_dir=/kaggle/working/yolo_training_results
!yolo settings datasets_dir=/kaggle/working/dataset_yolo

✅ Updated 'runs_dir=/kaggle/working/yolo_training_results'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/kaggle/working/datasets",
  "weights_dir": "weights",
  "runs_dir": "/kaggle/working/yolo_training_results",
  "uuid": "1bfc3e992d24318da58ddee183be5bf9388a31f26bab1738e986ec4d297417ff",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings
✅ Updated 'datasets_dir=/kaggle/working/dataset_yolo'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/kaggle/working/dataset_yolo",
  "weights_dir": "weights",
  "runs_dir": "/kaggle/working/yolo_training_results",
  

In [5]:
# Define base directories
BASE_DIR = "/kaggle/working/yolo_training_results"
PREDICTIONS_DIR = "/kaggle/working/yolo_validation_predictions"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# Dataset path
DATASET_PATH = "/kaggle/working/RasoulDataset"

In [6]:
# Generating yaml file
# Define YOLO dataset structure
data = {
    'train': '/kaggle/working/dataset/images/train',  # Path to training images
    'val': '/kaggle/working/dataset/images/val',  # Path to validation images
    'test': '/kaggle/working/dataset/images/test', # Path to test images
    'nc': 1,  # Number of classes
    'names': ['ICH']  # Class names
}

# Save YAML file
yaml_path = "/kaggle/working/dataset/dataset.yaml"

with open(yaml_path, 'w') as file:
    yaml.dump(data, file, default_flow_style=False)

print(f"YAML file saved at: {yaml_path}")

YAML file saved at: /kaggle/working/dataset/dataset.yaml


In [7]:
# Set the seed value
SEED = 42  

# Python’s built-in random module
random.seed(SEED)

# NumPy
np.random.seed(SEED)

# PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Ensure deterministic behavior in PyTorch
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set PYTHONHASHSEED to ensure consistent behavior in hash-based operations
os.environ["PYTHONHASHSEED"] = str(SEED)

In [8]:
# Model configurations (as a list to maintain order)
models = [
    "yolov8n-obb"
]

# Training parameters
EPOCHS = 50
BATCH_SIZE = 32

# Store results for visualization
results_dict = {}

# Values to try
vals = [0.2, 0.5, 1.0]

In [9]:
# Loop through each model
for model_name in models:
    for val in vals:

        print(f"\033[1m\n🚀 Training started for {model_name} with {val}...\033[0m\n")
        torch.cuda.empty_cache()
        gc.collect()

        model = YOLO(f"{model_name}.pt")

        # Define a unique name for each rotation setting
        run_name = f"{model_name}_mos{val}"

        # Train the model
        model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            device=[0,1],
            augment=False,
            seed=SEED,
            project=BASE_DIR,
            name=run_name,
            save=True,
            cache=True,
            val=True,
            save_txt=True,
            save_conf=True,

            # Augmentations
            degrees=0.0,              
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
            translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
            fliplr=0.0, flipud=0.0,
            mosaic=val,   # ✅
            mixup=0.0, copy_paste=0.0, auto_augment=None,
            erasing=0.0
            )

        # Updated result_dir
        result_dir = os.path.join(BASE_DIR, run_name)

        # Store CSV results
        results_dict[run_name] = os.path.join(result_dir, "results.csv")

        del model
        torch.cuda.empty_cache()
        gc.collect()


🚀 Training started for yolov8n-obb with 0.2...

Ultralytics 8.3.199 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
                                                        CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=None, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=0.2, multi_scale=False, name=yol